In [30]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import torch.optim as optim
import torch.nn as nn
from core.benchmarks import *
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
from core.benchmarks import *
import pandas as pd
from core.CVsplits import *
import logging
from core.Log import *
import json
OUTER_FOLDS = 4; INNER_FOLDS = 3

logging.shutdown()
setup_logger('INNER_train')
##['INNER_train', 'OUTER_train', 'OUTER_evaluate', 'Close']
log = logging.getLogger('INNER_train')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Create Holdout dataset for FINAL Model Evaluation
from sklearn.model_selection import train_test_split
from core.Log import load_dataset_info, save_dataset_info


main_dataset = load_dataset_info(file="data/data_info.json")
labels = [lbl['label'] for lbl in main_dataset]
main_training, final_test = train_test_split(main_dataset,
											 test_size=33,     # 33 for 4 OUTER folds
											 stratify=labels,
											 random_state=42)
print(len(final_test))

for sample in main_dataset:
	if sample in final_test: sample['pool'] = 'holdout'
	else: sample['pool'] = 'main'
#save_dataset_info(main_dataset, file="data/data_info.json")


In [ ]:
pool = [i['pool'] for i in main_dataset]
print(f"Total samples, {len(main_dataset)}, Normal cases: {labels.count(0)}, Takotsubo Cases: {labels.count(1)}")
print(f"Main cases: {pool.count('main')}, Holdout Cases: {pool.count('holdout')} ")
pools = list(set(pool))
for p in pools:
	lbls_in_pool = [d['label'] for d in main_dataset if d['pool'] == p]
	print(f"Pool: {p}, Total samples: {len(lbls_in_pool)}, Normal cases: {lbls_in_pool.count(0)}, Takotsubo Cases: {lbls_in_pool.count(1)}")


In [ ]:
from core.CVsplits import create_folds_stats

create_folds_stats(main_dataset, OUTER_K=4, INNER_K=3)


In [2]:
import itertools
import json
## 1. Define all model-specific hyperparameter sweeps in one dictionary
model_configs = {
#	"MultiViewCNN": {
#		"LR_SWEEP": [2e-4, 5e-4, 8e-4],
#		"DR_SWEEP": [0.2, 0.3, 0.4],
#		"WD_SWEEP": [1e-5]
#	},


	#"RN18+MLP": {
	#	"LR_SWEEP": [2e-4, 5e-4, 8e-6],
	#	"DR_SWEEP": [0.2, 0,3, 0.4],
	#	"WD_SWEEP": [1e-4, 1e-6]
	#},
	"Axial":    {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4, 1e-6], "DR_SWEEP": [0.3, 0.2]},
	"Coronal":  {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4, 1e-6], "DR_SWEEP": [0.3, 0.2]},
	"Sagittal": {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4, 1e-6], "DR_SWEEP": [0.3, 0.2]},
	}

# 2. Define global parameters that are the same for all models
GLOBAL_PARAMS = {
	"P": 4,
	"Epochs": 30,
}

INNER_CV_parameters = []
ID = 1

# Iterate through each model and its specific configuration
for model_name, config in model_configs.items():

	# Generate all unique combinations of the model's hyperparameters
	# e.g., for MultiViewCNN, this will create (1e-3, 0.3, 1e-4), (1e-3, 0.4, 1e-4), etc.
	hp_combinations = list(itertools.product(
		config['LR_SWEEP'],
		config['DR_SWEEP'],
		config['WD_SWEEP']
	))

	# Loop through outer and inner folds
	for outer_fold_idx in range(1, 5):
		for inner_fold_idx in range(1, 4):
			for i, (lr, dr, wd) in enumerate(hp_combinations):
				#item = { "Model": model_name, 'OUTER_FOLD': outer_fold_idx, 'INNER_FOLD': inner_fold_idx, "HPset": i+19, "LR": lr, "WD": wd, "DR": dr, "P": GLOBAL_PARAMS['P'], "Epochs": GLOBAL_PARAMS['Epochs'], "trained": False }
				#print(i+1)
				item = {"Model": model_name,
						"OUTER_FOLD": outer_fold_idx,
						"INNER_FOLD": inner_fold_idx,
						"HPset": i+1,
						"LR": lr,
						"WD": wd,
						"DR": dr,
						"P": GLOBAL_PARAMS['P'],
						"Epochs": GLOBAL_PARAMS['Epochs'],
						"trained": False}
				print(item)
				INNER_CV_parameters.append(item)
				ID += 1

print(f"Total combinations generated: {len(INNER_CV_parameters)}")

with open("NCV_4_3_folds/single_inner_experiments.json", "w") as f:
	json.dump(INNER_CV_parameters, f, indent=2)



{'Model': 'Axial', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 1, 'LR': 0.0002, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
{'Model': 'Axial', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 2, 'LR': 0.0002, 'WD': 1e-06, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
{'Model': 'Axial', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 3, 'LR': 0.0002, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 30, 'trained': False}
{'Model': 'Axial', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 4, 'LR': 0.0002, 'WD': 1e-06, 'DR': 0.2, 'P': 4, 'Epochs': 30, 'trained': False}
{'Model': 'Axial', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 5, 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
{'Model': 'Axial', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 6, 'LR': 0.0005, 'WD': 1e-06, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
{'Model': 'Axial', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 7, 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 30, 'trained': Fa

In [4]:
INNER_CV_parameters = load_from_json("NCV_4_3_folds/single_inner_experiments.json")
df = pd.DataFrame(INNER_CV_parameters)
#df.to_csv("NCV/single_inner_experiments.csv", index=False)
df


Loaded NCV_4_3_folds/single_inner_experiments.json.


,Model,OUTER_FOLD,INNER_FOLD,HPset,LR,WD,DR,P,Epochs,trained
0,Axial,1,1,1,0.0002,0.000100,0.3,4,30,False
1,Axial,1,1,2,0.0002,0.000001,0.3,4,30,False
2,Axial,1,1,3,0.0002,0.000100,0.2,4,30,False
3,Axial,1,1,4,0.0002,0.000001,0.2,4,30,False
4,Axial,1,1,5,0.0005,0.000100,0.3,4,30,False
...,...,...,...,...,...,...,...,...,...,...
283,Sagittal,4,3,4,0.0002,0.000001,0.2,4,30,False
284,Sagittal,4,3,5,0.0005,0.000100,0.3,4,30,False
285,Sagittal,4,3,6,0.0005,0.000001,0.3,4,30,False
286,Sagittal,4,3,7,0.0005,0.000100,0.2,4,30,False


In [31]:

INNER_CV_parameters = load_from_json("NCV_4_3_folds/single_inner_experiments.json")
filtered = [
	exp for exp in INNER_CV_parameters
	if exp["Model"] == "Sagittal"       # Sagittal, Coronal
	#and exp["OUTER_FOLD"] == 1       # 2, 3, 4
	#and exp["INNER_FOLD"] == 1       # 2, 3,
	and exp["HPset"] == 1
	and exp["trained"] == False
]
f"experiments: {len(filtered)}"


Loaded NCV_4_3_folds/single_inner_experiments.json.


'experiments: 5'

In [32]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score

def append_experiment_results(item, path="NCV/single_inner_summary.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")

def train_INNER_model(model, train_loader, val_loader, experiment):
	log = logging.getLogger('INNER_train')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	epochs = experiment['Epochs']
	model_name  = experiment['Model']
	out=experiment['OUTER_FOLD']
	inn=experiment['INNER_FOLD']
	HP = experiment['HPset']
	LR = experiment['LR']
	WD = experiment['WD']
	DR = experiment['DR']
	P = experiment['P']
	TH = 0.5
	rel_thresh = 5e-3 if np.isclose(LR, 5e-4) else 3e-3
	sch_patience, sch_cooldown = 2, 1

	ES_PATIENCE = max(P, sch_patience + sch_cooldown + 2)
	alpha_ema = 0.30  # smoothing for EMA of val loss

	optimizer = optim.Adam(model.parameters(), lr= LR, weight_decay=WD)
	scheduler = ReduceLROnPlateau(optimizer, mode='min',
								  patience=sch_patience, factor=0.5,
								  threshold=rel_thresh, threshold_mode='rel',
								  cooldown=sch_cooldown, min_lr=1e-6)
	criterion = nn.BCEWithLogitsLoss()

	# --- best trackers ---
	best_val_loss = np.inf
	best_epoch    = -1
	best_lr_at_best = LR
	auc_at_best   = np.nan
	accuracy_at_best = np.nan

	# --- EMA & patience ---
	ema_val = None
	best_ema = np.inf
	no_improve = 0

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)
	#log.info("Model;ExpID;OuterFold;HPset;Epoch;TrainLoss;TrainAcc;ValLoss;ValAcc;AUC;Brier;EMA_ValLoss;LR;NoImprove;LrDrop;EsTriggered;BestValLoss;BestEpoch;WallTimeSec\n")

	print(f" train_N: {train_N}, val_N: {val_N}		↳ Training model... ")
	for epoch in range(epochs):
		model.train()
		running_loss = 0.0
		y_true, y_pred = [], []
		for batch in train_loader:
			#axi = batch["axial_image"].to(device)
			#cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)

			optimizer.zero_grad(set_to_none=True)

			#logits = model(axi, met)
			#logits = model(cor, met)
			logits = model(sag, met)
			T_loss = criterion(logits, lbl)

			T_loss.backward()
			optimizer.step()

			running_loss += T_loss.item() * lbl.size(0)
			with torch.no_grad():
				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()
				y_true.append(lbl.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		TrainLoss = running_loss / max(1, train_N)
		y_true = np.concatenate(y_true).reshape(-1)
		y_pred = np.concatenate(y_pred).reshape(-1)
		TrainAcc  = (y_true == y_pred).mean()

		model.eval()
		running_loss = 0.0
		y_true, y_prob, y_pred = [], [], []

		with torch.no_grad():
			for batch in val_loader:
				#axi = batch["axial_image"].to(device)
				#cor = batch["coronal_image"].to(device)
				sag = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				#logits = model(axi, met)
				#logits = model(cor, met)
				logits = model(sag, met)
				V_loss = criterion(logits, lbl)
				running_loss += V_loss.item() * lbl.size(0)

				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()

				y_true.append(lbl.detach().cpu().numpy())
				y_prob.append(probs.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		ValLoss = running_loss / max(1, val_N)

		y_true  = np.concatenate(y_true).reshape(-1)
		y_prob  = np.concatenate(y_prob).reshape(-1)
		y_pred  = np.concatenate(y_pred).reshape(-1)

		ValAcc  = (y_true == y_pred).mean()
		AUC = roc_auc_score(y_true, y_prob)

		# -------------- EMA + scheduler --------------
		ema_val = ValLoss if ema_val is None else alpha_ema*ValLoss + (1 - alpha_ema)*ema_val
		prev_lr = optimizer.param_groups[0]['lr']
		scheduler.step(ema_val)  # schedule on EMA, not raw ValLoss
		new_lr = optimizer.param_groups[0]['lr']
		lr_drop = int(new_lr < prev_lr)

		# -------------- early stopping test -----------
		improved = ema_val < best_ema * (1 - rel_thresh)
		if improved:
			best_ema = ema_val
			best_val_loss = ValLoss
			best_epoch = epoch
			best_lr_at_best = new_lr
			accuracy_at_best = ValAcc
			auc_at_best = AUC
			no_improve = 0
		else:
			no_improve += 1

		es_triggered = int(no_improve >= ES_PATIENCE)
		line=f"{model_name};{out};{inn};{HP};{epoch};{TrainLoss};{TrainAcc};{ValLoss};{ValAcc};{AUC};{ema_val};{new_lr};{no_improve};{lr_drop};{es_triggered};{best_val_loss};{best_epoch}"
		log.info(line)
		if es_triggered: break

# ----- final return (best state + summary for outer fold) -----
	summary = {
        "Model": model_name,
        "OuterFold": out,
        "Innerfold": inn,
		"DR": DR,
        "LR": LR,
        "WD": WD,
        "BestEpoch": best_epoch,
        "BestValLoss": float(best_val_loss),
        "AUC_at_Best": float(auc_at_best) if auc_at_best is not None else np.nan,
		"Accuracy_at_Best": float(accuracy_at_best),
        "LR_at_Best": float(best_lr_at_best),
        "ES_Patience_Used": ES_PATIENCE,
        "RelThresh": rel_thresh,
        "EMA_alpha": alpha_ema,
    }
	return summary



In [33]:
main_dataset = load_dataset_info(file="data/data_info.json")
DL = DataLoaderFactory(main_dataset)
log = logging.getLogger('INNER_train')

#log.info("Model;OuterFold;InnerFold;HPset;Epoch;TrainLoss;TrainAcc;ValLoss;ValAcc;AUC;EMA_ValLoss;LR;NoImprove;LrDrop;EsTriggered;BestValLoss;BestEpoch")


In [28]:
#INNER_CV_parameters = load_from_json("NCV_4_3_folds/INNER_experiments.json")
INNER_CV_parameters = load_from_json("NCV_4_3_folds/single_inner_experiments.json")
def is_completed(exp):
    return (
        exp.get("Model") == "Sagittal"       # Sagittal, Coronal
        and exp.get("OUTER_FOLD") in {3}
        and exp.get("INNER_FOLD") == 1
        and exp.get("HPset") in {1}
    )

updated = 0
for exp in INNER_CV_parameters:
    if is_completed(exp):
        exp["trained"] = True          # normalize to lowercase
        updated += 1
print(f"Marked {updated} experiments as trained=True.")

with open("NCV_4_3_folds/single_inner_experiments.json", "w") as f:
    json.dump(INNER_CV_parameters, f, indent=2)
print("single_inner_experiments.json updated.")


Loaded NCV_4_3_folds/single_inner_experiments.json.
Marked 1 experiments as trained=True.
single_inner_experiments.json updated.


In [36]:
INNER_CV_parameters = load_from_json("NCV_4_3_folds/single_inner_experiments.json")
filtered = [
	exp for exp in INNER_CV_parameters
	if exp["Model"] == "Coronal"       # Sagittal, Coronal
	#and exp["OUTER_FOLD"] == 1       # 2, 3, 4
	#and exp["INNER_FOLD"] == 1       # 2, 3,
	and exp["HPset"] == 1
	and exp["trained"] == False
]
print(f"experiments: {len(filtered)}")
#REMEMBER: RERUN TRAINING CELL TO ADD DR TO SUMMARY


Loaded NCV_4_3_folds/single_inner_experiments.json.
experiments: 12


In [37]:

for experiment in filtered:
	print(experiment)
	OUT = experiment['OUTER_FOLD']
	INN = experiment['INNER_FOLD']
	train_loader, val_loader = DL.create_inner_loadersSingleView(OUT-1, INN-1)
	DR = experiment['DR']
	model = SingleViewClassifier(DR)
	results = train_INNER_model(model, train_loader, val_loader, experiment)
	append_experiment_results(results)



{'Model': 'Coronal', 'OUTER_FOLD': 1, 'INNER_FOLD': 1, 'HPset': 1, 'LR': 0.0002, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
 train_N: 62, val_N: 31		↳ Training model... 
{'Model': 'Coronal', 'OUTER_FOLD': 1, 'INNER_FOLD': 2, 'HPset': 1, 'LR': 0.0002, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
 train_N: 62, val_N: 31		↳ Training model... 
{'Model': 'Coronal', 'OUTER_FOLD': 1, 'INNER_FOLD': 3, 'HPset': 1, 'LR': 0.0002, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
 train_N: 62, val_N: 31		↳ Training model... 
{'Model': 'Coronal', 'OUTER_FOLD': 2, 'INNER_FOLD': 1, 'HPset': 1, 'LR': 0.0002, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
 train_N: 62, val_N: 31		↳ Training model... 
{'Model': 'Coronal', 'OUTER_FOLD': 2, 'INNER_FOLD': 2, 'HPset': 1, 'LR': 0.0002, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
 train_N: 62, val_N: 31		↳ Training model... 
{'Model': 'Coronal', 'OUTER_FOLD': 2, 'I